# クラスタ3「経済・物価・賃金」専用分析

クラスタ0・4除外後の再正規化結果から、クラスタ3だけを取り出して構成、年次推移、主要話者、代表発言、ランダム標本を確認します。クラスタリングの再学習や割当変更は行いません。

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if not (ROOT / 'analysis').exists():
    ROOT = ROOT.parent
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

INPUT_DIR = ROOT / 'analysis/clustering_2016_2026_nouns_excluding_0_4_renormalized'
OUTPUT_DIR = ROOT / 'analysis/cluster_3_economy'
CLUSTER_ID = 3
EXPECTED_LABEL = '経済・物価・賃金'
RANDOM_STATE = 42
SAMPLE_SIZE = 30

sns.set_theme(style='whitegrid')
mac_font_paths = list(Path('/System/Library/Fonts').glob('*角*W3.ttc'))
if not mac_font_paths:
    raise FileNotFoundError('macOSのヒラギノ角ゴシック W3が見つかりません')
font_manager.fontManager.addfont(str(mac_font_paths[0]))
MAC_JAPANESE_FONT = font_manager.FontProperties(fname=str(mac_font_paths[0])).get_name()
plt.rcParams['font.family'] = MAC_JAPANESE_FONT
plt.rcParams['axes.unicode_minus'] = False
print(f'Matplotlib font: {MAC_JAPANESE_FONT}')

## 1. クラスタ3の抽出と検証

In [ ]:
all_speeches = pd.read_csv(INPUT_DIR / 'speech_clusters_excluding_0_4.csv', encoding='utf-8-sig', parse_dates=['date'])
cluster_summary = pd.read_csv(INPUT_DIR / 'cluster_summary_renormalized.csv', encoding='utf-8-sig')
required = {'speechID', 'date', 'year', 'speaker', 'speakerPosition', 'speech', 'speech_cleaned', 'review_flags', 'speechURL', 'cluster', 'cluster_label', 'centroid_distance', 'representativeness'}
missing = required - set(all_speeches.columns)
assert not missing, f'必要な列がありません: {sorted(missing)}'
assert all_speeches['speechID'].is_unique
cluster3 = all_speeches.loc[all_speeches['cluster'].eq(CLUSTER_ID)].copy().reset_index(drop=True)
assert len(cluster3) == 4430
assert cluster3['cluster_label'].eq(EXPECTED_LABEL).all()
assert cluster3['speechID'].is_unique
summary_row = cluster_summary.loc[cluster_summary['cluster'].eq(CLUSTER_ID)].iloc[0]
display(pd.Series({
    'クラスタ番号': CLUSTER_ID,
    'クラスタ名': EXPECTED_LABEL,
    '発言数': len(cluster3),
    '再正規化後の割合': f"{summary_row['ratio']:.2%}",
    '対象年': f"{cluster3['year'].min()}〜{cluster3['year'].max()}",
    '話者数': cluster3['speaker'].nunique(),
    '上位特徴語': summary_row['top_terms'],
}).to_frame('value'))

## 2. 年別推移

In [ ]:
all_year_counts = all_speeches.groupby('year').size().rename('all_retained_speeches')
cluster_year_counts = cluster3.groupby('year').size().rename('cluster3_speeches')
yearly = (
    pd.concat([all_year_counts, cluster_year_counts], axis=1)
    .fillna(0)
    .astype({'all_retained_speeches': 'int64', 'cluster3_speeches': 'int64'})
    .assign(share_within_year=lambda d: d['cluster3_speeches'] / d['all_retained_speeches'])
    .reset_index()
)
assert len(yearly) == 11
display(yearly.style.format({'share_within_year': '{:.1%}'}))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=yearly, x='year', y='cluster3_speeches', color='#4C78A8', ax=axes[0])
axes[0].set(title='クラスタ3の年別発言数', xlabel='年', ylabel='発言数')
sns.lineplot(data=yearly, x='year', y='share_within_year', marker='o', color='#E45756', ax=axes[1])
axes[1].yaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
axes[1].set(title='各年の残存発言に占めるクラスタ3の割合', xlabel='年', ylabel='割合')
plt.tight_layout();

## 3. 全期間の月別推移

データの最初の月から取得最終月までを連続した月系列にします。発言がない月は件数0、母数となる残存発言もない月の割合は欠損です。

In [ ]:
first_month = all_speeches['date'].min().to_period('M')
last_month = all_speeches['date'].max().to_period('M')
full_months = pd.period_range(first_month, last_month, freq='M')
all_month_counts = all_speeches.groupby(all_speeches['date'].dt.to_period('M')).size().reindex(full_months, fill_value=0)
cluster_month_counts = cluster3.groupby(cluster3['date'].dt.to_period('M')).size().reindex(full_months, fill_value=0)
monthly = pd.DataFrame({
    'month': full_months.astype(str),
    'month_start': full_months.to_timestamp(),
    'all_retained_speeches': all_month_counts.to_numpy(dtype='int64'),
    'cluster3_speeches': cluster_month_counts.to_numpy(dtype='int64'),
})
monthly.loc[:, 'share_within_month'] = (
    monthly['cluster3_speeches'] / monthly['all_retained_speeches'].where(monthly['all_retained_speeches'].gt(0))
)
assert monthly['cluster3_speeches'].sum() == len(cluster3)
assert monthly['month'].is_unique
display(monthly.style.format({'share_within_month': '{:.1%}', 'month_start': '{:%Y-%m-%d}'}))
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
sns.lineplot(data=monthly, x='month_start', y='cluster3_speeches', color='#4C78A8', ax=axes[0])
axes[0].fill_between(monthly['month_start'], monthly['cluster3_speeches'], color='#4C78A8', alpha=.2)
axes[0].set(title='クラスタ3の月別発言数', xlabel='', ylabel='発言数')
sns.lineplot(data=monthly, x='month_start', y='share_within_month', color='#E45756', ax=axes[1])
axes[1].yaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[1].set(title='各月の残存発言に占めるクラスタ3の割合', xlabel='年月', ylabel='割合')
plt.tight_layout();

## 4. 主要話者・役職・レビュー旗・本文長

In [ ]:
top_speakers = cluster3['speaker'].value_counts().head(20).rename_axis('speaker').reset_index(name='count')
top_positions = cluster3['speakerPosition'].fillna('役職なし').replace('', '役職なし').value_counts().head(20).rename_axis('speakerPosition').reset_index(name='count')
review_flags = cluster3['review_flags'].fillna('なし').replace('', 'なし').value_counts().rename_axis('review_flags').reset_index(name='count')
text_length = cluster3['speech_cleaned'].str.len()
display(top_speakers, top_positions, review_flags)
display(text_length.describe(percentiles=[.1, .25, .5, .75, .9]).rename('characters').to_frame())
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=top_speakers.head(15), y='speaker', x='count', color='#72B7B2', ax=axes[0])
axes[0].set(title='主要話者（上位15人）', xlabel='発言数', ylabel='話者')
sns.histplot(text_length.clip(upper=text_length.quantile(.99)), bins=40, color='#F58518', ax=axes[1])
axes[1].set(title='クリーニング本文長（上位1%をクリップ）', xlabel='文字数', ylabel='発言数')
plt.tight_layout();

## 5. 重心に近い代表発言10件

In [ ]:
representatives = cluster3.nsmallest(10, 'centroid_distance').copy()
display(representatives[['speechID', 'date', 'speaker', 'speakerPosition', 'centroid_distance', 'speech_cleaned', 'speechURL']])

## 6. ランダム30発言の内容確認

In [ ]:
sample = cluster3.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).copy()
sample.insert(0, 'sample_order', range(1, SAMPLE_SIZE + 1))
sample.loc[:, 'review_decision'] = ''
sample.loc[:, 'review_notes'] = ''
assert len(sample) == SAMPLE_SIZE and sample['speechID'].is_unique
for row in sample.itertuples(index=False):
    position = row.speakerPosition if pd.notna(row.speakerPosition) and str(row.speakerPosition).strip() else '役職なし'
    flags = row.review_flags if pd.notna(row.review_flags) and str(row.review_flags).strip() else 'なし'
    cleaned = str(row.speech_cleaned).replace('\n', '  \n')
    original = str(row.speech).replace('\n', '  \n')
    display(Markdown(
        f"### {row.sample_order}. {row.speaker}（{position}）— {row.date:%Y-%m-%d}\n"
        f"- 発言ID: `{row.speechID}`\n"
        f"- レビューフラグ: {flags}\n"
        f"- 重心距離: {row.centroid_distance:.6f}\n"
        f"- [国会会議録の発言ページ]({row.speechURL})\n\n"
        f"**クリーニング本文**\n\n{cleaned}\n\n"
        f"<details><summary>原文を表示</summary>\n\n{original}\n\n</details>"
    ))

## 7. 分析表とレビュー標本の出力

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cluster3.to_csv(OUTPUT_DIR / 'cluster_3_speeches.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
yearly.to_csv(OUTPUT_DIR / 'cluster_3_yearly.csv', index=False, encoding='utf-8-sig')
monthly.to_csv(OUTPUT_DIR / 'cluster_3_monthly.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
top_speakers.to_csv(OUTPUT_DIR / 'cluster_3_top_speakers.csv', index=False, encoding='utf-8-sig')
representatives.to_csv(OUTPUT_DIR / 'cluster_3_representative_10.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
sample.to_csv(OUTPUT_DIR / 'cluster_3_random_30.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
assert len(pd.read_csv(OUTPUT_DIR / 'cluster_3_speeches.csv', encoding='utf-8-sig')) == len(cluster3)
assert pd.read_csv(OUTPUT_DIR / 'cluster_3_monthly.csv', encoding='utf-8-sig')['cluster3_speeches'].sum() == len(cluster3)
assert len(pd.read_csv(OUTPUT_DIR / 'cluster_3_random_30.csv', encoding='utf-8-sig')) == SAMPLE_SIZE
print(f'出力先: {OUTPUT_DIR}')
print(f'検証完了: クラスタ3 {len(cluster3):,}件、代表発言{len(representatives)}件、ランダム標本{len(sample)}件')